# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yara-08/ML-FlyRank-Internship2/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

> **Rule:** Recommend refreshing articles that have not been updated for a long time and still receive meaningful search visibility. Staleness indicates that content may need updating, while visibility suggests that improving the page could benefit more users.

> **Signal checks:**
> * **Staleness:** *MIXED*. Pages updated 91–180 days ago showed the highest decline rate (61.1%), while the 181+ day group did not continue this pattern. The oldest bucket also contained only 174 pages, so the evidence is limited.
> * **Visibility:** *MIXED*. Decline rates did not consistently increase with impression tier. However, higher visibility remains useful for prioritizing refresh opportunities because improvements can have greater user impact.

> **Reason code**: *STALE_VISIBLE*


> **Action label**: *REFRESH_CONTENT*

In [72]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
df = pd.read_csv("/content/ML-FlyRank-Internship2/data/raw/content_refresh_anonymized.csv")

# -------------------------------
# Signal check 1: Staleness
# -------------------------------
freshness_check = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          declining_pct=("trend_direction",
                         lambda x: (x == "down").mean() * 100)
      )
      .round(1)
)

# -------------------------------
# Signal check 2: Visibility
# -------------------------------
visibility_check = (
    df.groupby("impression_tier")
      .agg(
          n=("content_id", "count"),
          declining_pct=("trend_direction",
                         lambda x: (x == "down").mean() * 100)
      )
      .round(1)
)

print("=== Signal Check 1: Freshness vs Decline ===")
display(freshness_check)
print("Verdict: MIXED")

print("\n=== Signal Check 2: Impression Tier vs Decline ===")
display(visibility_check)
print("Verdict: MIXED")

=== Signal Check 1: Freshness vs Decline ===


,n,declining_pct
freshness_tier,,
0-30,20480,51.1
181+,174,47.1
31-90,175,58.9
91-180,9171,61.1


Verdict: MIXED

=== Signal Check 2: Impression Tier vs Decline ===


,n,declining_pct
impression_tier,,
excellent,1078,46.2
good,7205,58.6
low,11248,45.4
moderate,10469,61.5


Verdict: MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

> **Baseline scoring rule:**
> Each article receives a transparent baseline score based on two signals: freshness and search visibility. Freshness tiers (0–30, 31–90, 91–180, 181+) are converted into points from 0 to 3, and impression tiers (low, moderate, good, excellent) are also converted into points from 0 to 3. The baseline score is the sum of these two signals. Articles with a score of 4 or higher receive the reason code **STALE_VISIBLE** and the action label **REFRESH_CONTENT**. The ranked queue is then sorted by baseline score, with impressions used as a tie-breaker.

In [73]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import numpy as np

# -------------------------------
# Assign points to each signal
# -------------------------------

freshness_points = {
    "0-30": 0,
    "31-90": 1,
    "91-180": 2,
    "181+": 3,
}

visibility_points = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3,
}

df["freshness_score"] = df["freshness_tier"].map(freshness_points)
df["visibility_score"] = df["impression_tier"].map(visibility_points)

# -------------------------------
# Baseline score
# -------------------------------

df["baseline_score"] = (
    df["freshness_score"] +
    df["visibility_score"]
)

# -------------------------------
# Reason code & action
# -------------------------------

df["reason_code"] = np.where(
    df["baseline_score"] >= 4,
    "STALE_VISIBLE",
    ""
)

df["action_label"] = np.where(
    df["baseline_score"] >= 4,
    "REFRESH_CONTENT",
    ""
)

# -------------------------------
# Rank queue
# -------------------------------

ranked_queue = (
    df.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = ranked_queue.index + 1

# -------------------------------
# Export
# -------------------------------

output = ranked_queue[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "freshness_tier",
        "impression_tier",
        "days_since_last_update",
        "impressions_90d",
    ]
]

os.makedirs("work/outputs", exist_ok=True)

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False,
)

print(f"Pages flagged for review: {(output['baseline_score'] >= 4).sum()}")

print(f"Highest baseline score: {output['baseline_score'].max()}")

print(f"CSV saved to work/outputs/baseline_action_score.csv")

print("Top 20 ranked pages:")
display(output.head(20))

Pages flagged for review: 3544
Highest baseline score: 6
CSV saved to work/outputs/baseline_action_score.csv
Top 20 ranked pages:


,rank,content_id,baseline_score,reason_code,action_label,freshness_tier,impression_tier,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,6,STALE_VISIBLE,REFRESH_CONTENT,181+,excellent,194,61678
1,2,content_7368877ea310,6,STALE_VISIBLE,REFRESH_CONTENT,181+,excellent,194,59472
2,3,content_5fe46e04994d,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,517715
3,4,content_2dba2b1f9536,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,443434
4,5,content_2c2606c5d176,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,347399
5,6,content_cb112fce36be,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,309910
6,7,content_9532f197bbc8,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,309192
7,8,content_36ff89c8214e,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,295097
8,9,content_b28d1efd668f,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,286608
9,10,content_813e88069237,5,STALE_VISIBLE,REFRESH_CONTENT,91-180,excellent,104,233561


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

> I manually reviewed the highest-ranked pages produced by my baseline rule. For each page, I recorded the recommended action, the reason code, a confidence note based on the observed signals, and one situation that could make the recommendation incorrect. This review is intended to check whether the rule produces sensible recommendations before building a machine learning model. The notes are decision-support only and do not prove that a page should definitely be refreshed.

In [81]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


review = ranked_queue.head(20).copy()

# ---------- Why selected ----------
review["why_selected"] = (
    "Visible page (" +
    review["impressions_90d"].astype(str) +
    " impressions) not updated for " +
    review["days_since_last_update"].astype(str) +
    " days."
)

# ---------- Confidence ----------
review["confidence_note"] = "Medium"

review.loc[
    (review["days_since_last_update"] >= 180) &
    (review["impression_tier"] == "excellent"),
    "confidence_note"
] = "Very High"

review.loc[
    (review["days_since_last_update"] >= 90) &
    (review["impression_tier"] == "excellent"),
    "confidence_note"
] = "High"

# Lower confidence for suspicious engagement
review.loc[
    review["engagement_rate"] < 1,
    "confidence_note"
] = "Medium"

# ---------- What could make it wrong ----------
review["what_would_make_it_wrong"] = (
    "The page may already be accurate and not require updating."
)

review.loc[
    review["engagement_rate"] < 1,
    "what_would_make_it_wrong"
] = (
    "Very low engagement may reflect measurement limits rather than poor content."
)

review.loc[
    review["content_age_days"] < 180,
    "what_would_make_it_wrong"
] = (
    "The page is not very old overall, so another issue besides freshness may explain its performance."
)

display(
    review[[
        "content_id",
        "action_label",
        "reason_code",
        "why_selected",
        "confidence_note",
        "what_would_make_it_wrong"
    ]]
)

,content_id,action_label,reason_code,why_selected,confidence_note,what_would_make_it_wrong
0,content_cf56e2e2e282,REFRESH_CONTENT,STALE_VISIBLE,Visible page (61678 impressions) not updated f...,Medium,Very low engagement may reflect measurement li...
1,content_7368877ea310,REFRESH_CONTENT,STALE_VISIBLE,Visible page (59472 impressions) not updated f...,High,The page may already be accurate and not requi...
2,content_5fe46e04994d,REFRESH_CONTENT,STALE_VISIBLE,Visible page (517715 impressions) not updated ...,High,The page may already be accurate and not requi...
3,content_2dba2b1f9536,REFRESH_CONTENT,STALE_VISIBLE,Visible page (443434 impressions) not updated ...,High,The page may already be accurate and not requi...
4,content_2c2606c5d176,REFRESH_CONTENT,STALE_VISIBLE,Visible page (347399 impressions) not updated ...,High,The page may already be accurate and not requi...
5,content_cb112fce36be,REFRESH_CONTENT,STALE_VISIBLE,Visible page (309910 impressions) not updated ...,High,"The page is not very old overall, so another i..."
6,content_9532f197bbc8,REFRESH_CONTENT,STALE_VISIBLE,Visible page (309192 impressions) not updated ...,High,The page may already be accurate and not requi...
7,content_36ff89c8214e,REFRESH_CONTENT,STALE_VISIBLE,Visible page (295097 impressions) not updated ...,High,"The page is not very old overall, so another i..."
8,content_b28d1efd668f,REFRESH_CONTENT,STALE_VISIBLE,Visible page (286608 impressions) not updated ...,High,"The page is not very old overall, so another i..."
9,content_813e88069237,REFRESH_CONTENT,STALE_VISIBLE,Visible page (233561 impressions) not updated ...,High,"The page is not very old overall, so another i..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

> Some recommendations from the baseline rule may be false positives because the rule only considers freshness and visibility. A page can be old and highly visible while still being accurate or intentionally evergreen. Similarly, pages with unusually low engagement may require investigation before deciding to refresh them.
> I also checked for data leakage. The baseline rule only uses current descriptive features (days_since_last_update and impression_tier) and does not use future performance, labels, or product-generated flags. Therefore, this baseline is intentionally simple and explainable. A future machine learning model should learn more complex patterns and outperform this rule-based approach.

In [82]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Weak picks (lowest-confidence recommendations)

weak_picks = ranked_queue[
    (ranked_queue["engagement_rate"] < 1) |
    (ranked_queue["content_age_days"] < 180)
].copy()

weak_picks["possible_issue"] = ""

weak_picks.loc[
    weak_picks["engagement_rate"] < 1,
    "possible_issue"
] = "Very low engagement needs manual review."

weak_picks.loc[
    weak_picks["content_age_days"] < 180,
    "possible_issue"
] = "Content is not very old despite being flagged."

print("Potential weak picks:", len(weak_picks))
display(
    weak_picks.head(10)[[
        "content_id",
        "baseline_score",
        "engagement_rate",
        "content_age_days",
        "days_since_last_update",
        "impression_tier",
        "possible_issue"
    ]]
)

# -------------------------
# Leakage check
# -------------------------

features_used = [
    "days_since_last_update",
    "impression_tier"
]

print("\nFeatures used by the rule:")
print(features_used)

print("\nLeakage check:")
print("✓ No future-window variables used.")
print("✓ No labels or target-derived features used.")
print("✓ No product flags used.")
print("✓ Rule only uses observable current-page signals.")

Potential weak picks: 25141


,content_id,baseline_score,engagement_rate,content_age_days,days_since_last_update,impression_tier,possible_issue
0,content_cf56e2e2e282,6,0.84,231,194,excellent,Very low engagement needs manual review.
5,content_cb112fce36be,5,2.08,126,104,excellent,Content is not very old despite being flagged.
7,content_36ff89c8214e,5,1.68,144,104,excellent,Content is not very old despite being flagged.
8,content_b28d1efd668f,5,3.83,153,104,excellent,Content is not very old despite being flagged.
9,content_813e88069237,5,1.37,153,104,excellent,Content is not very old despite being flagged.
10,content_c21024970297,5,1.49,126,104,excellent,Content is not very old despite being flagged.
11,content_c8e9d6ab9013,5,0.00,362,104,excellent,Very low engagement needs manual review.
13,content_d17681677e69,5,0.65,313,104,excellent,Very low engagement needs manual review.
16,content_3d94572c3a35,5,5.22,124,104,excellent,Content is not very old despite being flagged.
18,content_33b4dceecad1,5,0.61,144,104,excellent,Content is not very old despite being flagged.



Features used by the rule:
['days_since_last_update', 'impression_tier']

Leakage check:
✓ No future-window variables used.
✓ No labels or target-derived features used.
✓ No product flags used.
✓ Rule only uses observable current-page signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.